# 05 — Crop-label distributions & model-readiness

Data cleaning / EDA on the training labels. Reads the scripted-pipeline output
`data/processed/training_crop_polygon.parquet` (one row per `COD_PREDIO`, a normalised
**list** of crops per polygon; see `CLAUDE.md` §5b).

This notebook:
1. Histogram of **all** crop entries (each crop in a multi-crop parcel counted separately).
2. How many parcels carry exactly **one** crop entry.
3. Histogram of crops among the **single-crop-only** parcels.
4. Notes on turning this into a model-ready label set.

Run with the project kernel: `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/05_crop_label_cleaning.ipynb`

In [ ]:
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROC = Path("../data/processed")
OUT = Path("../docs/figures")
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

gdf = gpd.read_parquet(PROC / "training_crop_polygon.parquet")
print(gdf.shape)
gdf[["COD_PREDIO", "crops", "crop_categories", "n_labels", "is_vegetated", "year", "area_ha"]].head()

## 1. Histogram of all crop entries

`crops` is a list per polygon. We **explode** it so an intercropped parcel like
`[CIRUELA, MANGO, PLATANO]` contributes one count to each of its three crops.

In [ ]:
def bar_top(counts, n, title, fname, color="#3b7dd8"):
    top = counts.head(n)[::-1]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(top))))
    ax.barh(top.index, top.values, color=color)
    total = counts.sum()
    for y, v in enumerate(top.values):
        ax.text(v, y, f" {v:,} ({v / total:.0%})", va="center", fontsize=8)
    ax.set_xlabel("count")
    ax.set_title(title)
    ax.margins(x=0.15)
    fig.tight_layout()
    fig.savefig(OUT / fname, bbox_inches="tight")
    plt.show()

In [ ]:
# Histogram of crop COMBINATIONS (pairs, triplets, ...) — the intercrop signatures.
# NOTE: `crops` comes back from parquet as a numpy ndarray, not a python list, so
# `isinstance(x, list)` is False and would silently leave every row a distinct (unhashable)
# array -> value_counts sees ~all combos as unique. Join on any non-str iterable instead.
def combo_key(x):
    return " + ".join(sorted(x)) if not isinstance(x, str) else x

combo = gdf.copy()
combo["combo"] = combo["crops"].apply(combo_key)

# Multi-crop parcels only (n_labels >= 2); single crops are the histograms above.
multi = combo[combo["n_labels"] >= 2]
combo_counts = multi["combo"].value_counts()
print(f"multi-crop parcels    : {len(multi):,} ({len(multi) / len(gdf):.1%} of all parcels)")
print(f"distinct combinations : {combo_counts.size:,}")
print("\ncombination size (crops per parcel):")
print(multi["n_labels"].value_counts().sort_index().to_string())

bar_top(combo_counts, 30, "Crop combinations (multi-crop parcels) — top 30",
        "hist_crop_combos.png", color="#9467bd")

In [ ]:
exploded = gdf.explode("crops", ignore_index=True)
# align the category list to the exploded crops for later filtering
cat_exploded = gdf[["crop_categories"]].explode("crop_categories", ignore_index=True)
exploded["category"] = cat_exploded["crop_categories"].values

all_counts = exploded["crops"].value_counts()
print(f"total crop entries : {len(exploded):,}")
print(f"distinct crops     : {all_counts.size:,}")
print(f"parcels            : {len(gdf):,}")
all_counts.head(25)

In [ ]:
bar_top(all_counts, 30, "All crop entries (multi-crop parcels counted per crop) — top 30", "hist_all_crops.png")

In [ ]:
# Long tail: how quickly does coverage saturate?
cum = all_counts.cumsum() / all_counts.sum()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cum) + 1), cum.values, color="#3b7dd8")
for thr in (0.90, 0.95, 0.99):
    k = int((cum < thr).sum()) + 1
    ax.axhline(thr, color="grey", lw=0.6, ls=":")
    ax.annotate(f"{thr:.0%} at {k} crops", (k, thr), fontsize=8, xytext=(5, -10),
                textcoords="offset points")
ax.set(xlabel="crop rank", ylabel="cumulative share of entries",
       title=f"Crop frequency is heavy-tailed: {all_counts.size} distinct labels")
ax.set_xlim(0, 60)
fig.tight_layout()
fig.savefig(OUT / "hist_all_crops_cumulative.png", bbox_inches="tight")
plt.show()

## 2. Parcels with exactly one crop entry

`n_labels == 1`. These are the clean single-label targets a first classifier should train on.

In [ ]:
n_single = int((gdf["n_labels"] == 1).sum())
print(f"single-crop parcels : {n_single:,} ({n_single / len(gdf):.1%})")
print(f"multi-crop parcels  : {len(gdf) - n_single:,} ({1 - n_single / len(gdf):.1%})")
print()
print("n_labels distribution:")
print(gdf["n_labels"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(6, 3.5))
nl = gdf["n_labels"].value_counts().sort_index()
ax.bar(nl.index.astype(str), nl.values, color="#3b7dd8")
for x, v in enumerate(nl.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.set(xlabel="crops per parcel (n_labels)", ylabel="parcels",
       title="Crops per parcel")
fig.tight_layout()
fig.savefig(OUT / "hist_n_labels.png", bbox_inches="tight")
plt.show()

## 3. Histogram of crops among single-crop-only parcels

The crop distribution restricted to `n_labels == 1` — the actual class prior a
single-label model would see.

In [ ]:
single = gdf[gdf["n_labels"] == 1].copy()
single["crop"] = single["crops"].str[0]
single["category"] = single["crop_categories"].str[0]
single_counts = single["crop"].value_counts()
print(f"single-crop parcels : {len(single):,}")
print(f"distinct crops here : {single_counts.size:,}")
bar_top(single_counts, 30, "Single-crop parcels only — top 30", "hist_single_crops.png", color="#2ca25f")

In [ ]:
# Same, but restricted to genuine crops (drop fallow / land_prep / pasture / unspecified).
crop_single = single[single["category"] == "crop"]
crop_counts = crop_single["crop"].value_counts()
print("single-crop parcels by category:")
print(single["category"].value_counts())
print(f"\nreal-crop single-label parcels: {len(crop_single):,}")
bar_top(crop_counts, 25, "Single-crop parcels, category == 'crop' only — top 25",
        "hist_single_realcrops.png", color="#2ca25f")

In [ ]:
# How many classes survive a minimum-support threshold (candidate model vocabulary)?
print("real-crop single-label classes with >= N parcels:")
for thr in (50, 100, 200, 300, 500):
    keep = crop_counts[crop_counts >= thr]
    print(f"  >= {thr:4d} parcels : {keep.size:2d} classes, "
          f"{keep.sum():,} parcels ({keep.sum() / len(crop_single):.0%} of real-crop)")
print("\ncandidate 12-class vocabulary (>= 300 parcels):")
print(list(crop_counts[crop_counts >= 300].index))

In [ ]:
# year distribution of single-crop parcels
crop_single = single[single["category"] == "crop"]
year_counts = crop_single["year"].value_counts()
print("single-crop parcels by year:")
print(year_counts)
print(f"\nreal-crop single-label parcels: {len(crop_single):,}")
# histogram of years using year_counts
year_counts = year_counts.sort_index()
fig = plt.figure(figsize=(6, 3.5))
year_counts.plot(kind="bar", ax=plt.gca(), color="#2ca25f")
plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Distribution of Single-Crop Parcels by Year")
plt.xticks(rotation=45)
# increase dpi
plt.gcf().set_dpi(300)
plt.tight_layout()

## 4. Getting this to a model-ready label set

Concrete steps (these become `labels.py` in the `src/` pipeline — see `plan.md` §7):

1. **Pick the grain.** Start with **single-crop parcels only** (`n_labels == 1`) for a clean
   baseline `y`; keep multi-crop parcels aside for a later multi-label head.
2. **Drop non-crop categories from the crop task.** Filter `category == 'crop'`. `DESCANSO`
   (fallow), `ARADO`/`GRADEO`/`MECANIZADO` (land-prep) and `PASTO*` (pasture) are **not crops** —
   either exclude them or model them as explicit land-cover classes (they help the model not
   misfile bare/prepared soil as a crop).
3. **Define a fixed class vocabulary + `other`.** Keep classes with enough support (e.g. the
   ~12 crops with ≥ 300 single-crop parcels); fold the long tail into `other` **or** drop it.
   Derive the vocabulary from **train folds only** (never the whole set) to avoid leakage.
4. **Handle imbalance.** ARROZ alone is ~60% of the real-crop single-label parcels. Use
   **macro-F1** (not accuracy) and class weights; never oversample before splitting.
5. **Quality flags, not silent drops.** Flag sub-pixel parcels (`area_ha < 0.09`, ~14%) and
   outsized ones (`area_ha > 50`, dissolve artifacts); flag but keep for inspection.
6. **Spatial split is mandatory.** ~86% of neighbours share a crop, so assign a `block_id`
   (10 km grid on EPSG:32717 centroids) and split by whole blocks. A random row split leaks.
7. **Freeze the label table.** Emit `modeling_parcels.parquet` = `COD_PREDIO, label, label_id,
   block_id, fold, split, area_ha, quality_ok, geometry` + `label_map.json`. Nothing downstream
   should re-read the raw `.dta/.xlsx`.

The cell below writes a first-pass single-label table so the numbers above are reproducible;
the production version (with splits + quality gate) belongs in `src/crop_classifier/labels.py`.

In [ ]:
VOCAB_MIN = 300
vocab = set(crop_counts[crop_counts >= VOCAB_MIN].index)

labels = crop_single[["COD_PREDIO", "crop", "year", "area_ha", "geometry"]].copy()
labels["label"] = np.where(labels["crop"].isin(vocab), labels["crop"], "other")
labels["area_pixels_est"] = labels["area_ha"] * 1e4 / 900  # 30 m Landsat pixels
labels["quality_flag_subpixel"] = labels["area_ha"] < 0.09
labels["quality_flag_oversized"] = labels["area_ha"] > 50

print(f"first-pass single-label table: {len(labels):,} parcels, {labels['label'].nunique()} classes")
print(f"sub-pixel flagged  : {labels['quality_flag_subpixel'].mean():.1%}")
print(f"oversized flagged  : {int(labels['quality_flag_oversized'].sum())} parcels")
print(labels["label"].value_counts())
# Not written to disk here — labels.py owns the canonical artifact (adds block_id/fold/split).
labels.head()

## 5. Signal check — Landsat scene availability over Piura (settles plan.md issue 1)

Before building any `features/` extraction we must know whether the historical Landsat archive
actually *has* enough acquisitions in our crop years. The bulk labels are **1998–99**, when:

- **Landsat 5 (TM)** downlink over South America depended on international ground stations and is
  known to be **sparse in the 1990s** — a real risk that many parcel-years have too few scenes to
  composite, regardless of cloud.
- **Landsat 7 (ETM+)** only launched **1999-04**, so 1998 is **L5-only**; and L7 is SLC-off (striped)
  after **2003-05**.
- The catastrophic **1997–98 El Niño** hit Piura directly → heavy cloud/flood contamination in
  exactly the highest-data years.

This section counts, per year over the Piura parcel AOI, (a) how many L5/L7/L8 scenes intersect it,
and — the number that actually matters — (b) the **median count of cloud-masked *valid monthly
observations* per parcel-year** on a small random parcel sample. If that median is well below ~6,
the p50 monthly time series is not buildable for most parcels and the modelling plan must change
(restrict to years/regions with coverage, widen the compositing window, or use sparse-series models
only).

Needs GEE auth (`uv run earthengine authenticate`, project `peru-crop-classifier`). Degrades to a
clean skip if Earth Engine is unavailable.

In [ ]:
import socket

socket.setdefaulttimeout(120)  # no GEE call can hang the run
GEE_PROJECT = "peru-crop-classifier"

EE_OK = False
try:
    import ee

    ee.Initialize(project=GEE_PROJECT)
    EE_OK = True
    print("Earth Engine initialised.")
except Exception as e:  # noqa: BLE001
    print(f"Earth Engine unavailable ({type(e).__name__}: {e}).")
    print("Run `uv run earthengine authenticate` then re-run this section. Skipping.")

# AOI = bounding box of all parcels (EPSG:4326). Scene intersection uses the rectangle;
# per-parcel valid-obs uses the actual polygons.
minx, miny, maxx, maxy = gdf.total_bounds
print(f"AOI bbox lon [{minx:.3f}, {maxx:.3f}]  lat [{miny:.3f}, {maxy:.3f}]")

# Years to probe: those that actually carry single-crop labels, focused on the bulk.
label_years = (
    crop_single["year"].dropna().astype(int).value_counts().sort_index()
)
PROBE_YEARS = [int(y) for y in label_years.index if label_years[y] >= 30]
print(f"probing {len(PROBE_YEARS)} years with >=30 single-crop parcels: {PROBE_YEARS}")

In [ ]:
if EE_OK:
    AOI = ee.Geometry.Rectangle([minx, miny, maxx, maxy])

    L5 = "LANDSAT/LT05/C02/T1_L2"
    L7 = "LANDSAT/LE07/C02/T1_L2"
    L8 = "LANDSAT/LC08/C02/T1_L2"

    def _missions(year):
        """Landsat missions carrying usable SR for a given year."""
        m = {}
        if year <= 2013:
            m["L5"] = L5
        if year >= 1999:
            m["L7"] = L7
        if year >= 2013:
            m["L8"] = L8
        return m

    def _mask_valid(img):
        """QA_PIXEL cloud/shadow/cirrus mask (C2 bits 1,2,3,4). Returns a 1-band 'val' image
        that is 1 where the pixel is clear, masked elsewhere."""
        qa = img.select("QA_PIXEL")
        clear = (
            qa.bitwiseAnd(1 << 1).eq(0)  # dilated cloud
            .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
            .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
        )
        return (
            ee.Image(1)
            .updateMask(clear)
            .rename("val")
            .copyProperties(img, ["system:time_start"])
        )

    def masked_coll(year):
        parts = [ee.ImageCollection(cid) for cid in _missions(year).values()]
        coll = parts[0]
        for p in parts[1:]:
            coll = coll.merge(p)
        y0 = ee.Date.fromYMD(year, 1, 1)
        return coll.filterBounds(AOI).filterDate(y0, y0.advance(1, "year")).map(_mask_valid)

    print("helpers ready")
else:
    print("skipped (no EE)")

In [ ]:
# --- Tier 1: raw scene counts intersecting the AOI, per mission per year ---
if EE_OK:
    rows = []
    for y in PROBE_YEARS:
        rec = {"year": y, "n_parcels": int(label_years.get(y, 0))}
        for name, cid in _missions(y).items():
            y0 = ee.Date.fromYMD(y, 1, 1)
            n = (
                ee.ImageCollection(cid)
                .filterBounds(AOI)
                .filterDate(y0, y0.advance(1, "year"))
                .size()
                .getInfo()
            )
            rec[name] = n
        rec["total_scenes"] = sum(rec.get(k, 0) for k in ("L5", "L7", "L8"))
        rows.append(rec)
        print(f"  {y}: {rec}")
    tier1 = pd.DataFrame(rows).set_index("year")
    # only missions that actually appear in the probed years (e.g. no L8 pre-2013)
    for m in ("L5", "L7", "L8"):
        if m not in tier1.columns:
            tier1[m] = 0
        tier1[m] = tier1[m].fillna(0).astype(int)
    display(tier1)
else:
    tier1 = None
    print("skipped (no EE)")

In [ ]:
# --- Tier 2: per-parcel cloud-masked valid observations, on a random sample per year ---
# The number that decides feasibility: for a real parcel in its own crop-year, how many
# (a) clear observations across the year and (b) distinct months have >=1 clear observation.
SAMPLE_PER_YEAR = 120
TIER2_YEARS = list(label_years.sort_values(ascending=False).head(6).index.astype(int))
TIER2_YEARS = [y for y in TIER2_YEARS if y in PROBE_YEARS]
rng = np.random.default_rng(0)


def to_fc(sample):
    feats = [
        ee.Feature(ee.Geometry(geom.__geo_interface__), {"cid": cid})
        for cid, geom in zip(sample["COD_PREDIO"], sample.geometry)
    ]
    return ee.FeatureCollection(feats)


if EE_OK:
    # Fully-masked, explicitly-typed base image so an empty month still yields a float 'val'
    # band (reducers on empty collections drop the band; selfMask gives an untyped MaskOnly).
    BASE = ee.ImageCollection(
        [ee.Image.constant(0).float().rename("val").updateMask(ee.Image.constant(0))]
    )
    t2rows = []
    for y in TIER2_YEARS:
        pool = crop_single[crop_single["year"] == y]
        take = min(SAMPLE_PER_YEAR, len(pool))
        sample = pool.iloc[rng.choice(len(pool), take, replace=False)]
        fc = to_fc(sample)
        coll = masked_coll(y).select("val").map(lambda im: im.float())

        # (a) annual valid-obs count per pixel -> max over parcel (best pixel, an upper bound)
        obs_img = coll.count().unmask(0).rename("valid_obs")

        # (b) valid months: 1 per month with any clear obs (BASE keeps the band on empty months)
        month_bands = []
        for m in range(1, 13):
            m0 = ee.Date.fromYMD(y, m, 1)
            mono = (
                coll.filterDate(m0, m0.advance(1, "month"))
                .merge(BASE)
                .sum()
                .unmask(0)
                .gt(0)
                .rename(f"m{m:02d}")
            )
            month_bands.append(mono)
        months_img = ee.Image.cat(month_bands).reduce(ee.Reducer.sum()).rename("valid_months")

        stack = obs_img.addBands(months_img)
        red = stack.reduceRegions(collection=fc, reducer=ee.Reducer.max(), scale=30).getInfo()
        vals = [f["properties"] for f in red["features"]]
        vo = np.array([v.get("valid_obs", 0) or 0 for v in vals], float)
        vm = np.array([v.get("valid_months", 0) or 0 for v in vals], float)
        t2rows.append(
            {
                "year": y,
                "n_sample": take,
                "median_valid_obs": np.median(vo),
                "median_valid_months": np.median(vm),
                "frac_ge6_months": float((vm >= 6).mean()),
                "frac_0_months": float((vm == 0).mean()),
            }
        )
        print(f"  {y}: {t2rows[-1]}")
    tier2 = pd.DataFrame(t2rows).set_index("year").sort_index()
    display(tier2)
else:
    tier2 = None
    print("skipped (no EE)")

In [ ]:
# --- Visualise + verdict ---
if EE_OK and tier1 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

    # (left) scene counts per mission per year, stacked, bars weighted by parcel count
    ax = axes[0]
    bottom = np.zeros(len(tier1))
    colors = {"L5": "#d95f02", "L7": "#7570b3", "L8": "#1b9e77"}
    for m in ("L5", "L7", "L8"):
        ax.bar(tier1.index.astype(str), tier1[m], bottom=bottom, label=m, color=colors[m])
        bottom += tier1[m].values
    ax.axhline(23, color="grey", ls=":", lw=0.8)  # ~1 mission's annual acquisitions (16-day)
    ax.set(xlabel="crop year", ylabel="scenes intersecting AOI",
           title="Landsat scenes over Piura AOI, by mission")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(title="mission")

    # (right) per-parcel valid months (the feasibility number)
    ax = axes[1]
    if tier2 is not None:
        ax.bar(tier2.index.astype(str), tier2["median_valid_months"], color="#3b7dd8")
        for x, v in enumerate(tier2["median_valid_months"].values):
            ax.text(x, v, f"{v:.0f}", ha="center", va="bottom", fontsize=8)
        ax.axhline(6, color="red", ls="--", lw=1, label="min 6/12 months")
        ax.set(xlabel="crop year", ylabel="median valid months / parcel",
               title="Cloud-masked valid months per parcel (sample)")
        ax.set_ylim(0, 12)
        ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "landsat_availability_piura.png", bbox_inches="tight")
    plt.show()

    # Verdict, weighted by where the labels actually are
    if tier2 is not None:
        w = label_years.reindex(tier2.index).fillna(0)
        wmed_months = float(np.average(tier2["median_valid_months"], weights=w)) if w.sum() else float("nan")
        wfrac6 = float(np.average(tier2["frac_ge6_months"], weights=w)) if w.sum() else float("nan")
        print("=== VERDICT (issue 1) ===")
        print(f"Parcel-weighted median valid MONTHS/parcel : {wmed_months:.1f} / 12")
        print(f"Parcel-weighted frac parcels with >=6 mo   : {wfrac6:.0%}")
        bulk = tier1.loc[tier1.index <= 2000, "total_scenes"]
        print(f"Total scenes in bulk years (<=2000)        : "
              f"{dict(bulk) if len(bulk) else 'n/a'}")
        if wmed_months >= 6:
            print("-> FEASIBLE: a 12-point monthly p50 series is buildable for most parcels.")
        elif wmed_months >= 3:
            print("-> MARGINAL: widen the compositing window (e.g. bi-monthly / seasonal) or "
                  "use sparse-series models; a dense 12-month series is not reliable.")
        else:
            print("-> NOT FEASIBLE as planned: archive too sparse in the bulk years. "
                  "Restrict to covered years/regions or rethink the temporal design.")
else:
    print("No EE results to plot — authenticate and re-run Section 5.")

### 5b. Gap structure — are the missing months scattered or clustered?

Coverage counts (§5) don't decide the interpolation question; **where** the gaps fall does. A
parcel missing 4 *scattered* months (good obs either side) interpolates almost losslessly; a parcel
missing a *contiguous* 4–5 month block (typical when El Niño cloud wipes out a season) would be
**extrapolated across unobserved phenology** — fine for evergreen perennials, but for a fast annual
(rice/maize/cotton) the whole flood→green→senescence cycle can live inside that gap, so
interpolation injects a wrong signal.

This measures, per parcel-year (same best-pixel monthly presence as §5), the **longest run of
consecutive missing months** (`max_gap`) and the number of separate gaps. The decision rule:
- most parcels `max_gap ≤ 2` → a monthly grid + light interpolation is safe;
- many parcels `max_gap ≥ 3` → bin coarser (**bi-monthly / seasonal**) so each cell stays a real
  measurement, and **abstain** on parcels whose largest gap is too big rather than filling it.

In [ ]:
# --- Gap structure per parcel-year: pull the 12-month presence vector, measure gap runs ---
def gap_metrics(pres):
    """pres: 1D array of 12 zeros/ones (1 = month has >=1 clear obs). Returns
    (n_obs, max_gap, n_gaps): longest run and number of runs of consecutive missing months."""
    max_gap = cur = n_gaps = 0
    for x in pres:
        if x < 0.5:
            cur += 1
            max_gap = max(max_gap, cur)
        else:
            if cur > 0:
                n_gaps += 1
            cur = 0
    if cur > 0:
        n_gaps += 1
    return int(pres.sum()), int(max_gap), int(n_gaps)


if EE_OK:
    rng2 = np.random.default_rng(7)
    grows = []
    per_year_maxgap = {}
    for y in TIER2_YEARS:
        pool = crop_single[crop_single["year"] == y]
        take = min(SAMPLE_PER_YEAR, len(pool))
        sample = pool.iloc[rng2.choice(len(pool), take, replace=False)]
        fc = to_fc(sample)
        coll = masked_coll(y).select("val").map(lambda im: im.float())

        # one band per month, max over parcel = "any clear pixel that month" (best-pixel, as §5)
        month_bands = []
        for m in range(1, 13):
            m0 = ee.Date.fromYMD(y, m, 1)
            month_bands.append(
                coll.filterDate(m0, m0.advance(1, "month"))
                .merge(BASE)
                .sum()
                .unmask(0)
                .gt(0)
                .rename(f"m{m:02d}")
            )
        stack = ee.Image.cat(month_bands)
        red = stack.reduceRegions(collection=fc, reducer=ee.Reducer.max(), scale=30).getInfo()

        maxgaps = []
        for f in red["features"]:
            p = f["properties"]
            pres = np.array([p.get(f"m{m:02d}", 0) or 0 for m in range(1, 13)], float)
            n_obs, max_gap, n_gaps = gap_metrics(pres)
            grows.append({"year": y, "n_obs": n_obs, "max_gap": max_gap, "n_gaps": n_gaps})
            maxgaps.append(max_gap)
        per_year_maxgap[y] = np.array(maxgaps)
        mg = per_year_maxgap[y]
        print(f"  {y}: median max_gap={np.median(mg):.0f}  "
              f"%(max_gap>=3)={np.mean(mg >= 3):.0%}  %(max_gap>=5)={np.mean(mg >= 5):.0%}")

    gaps = pd.DataFrame(grows)
    display(gaps.groupby("year").agg(
        median_n_obs=("n_obs", "median"),
        median_max_gap=("max_gap", "median"),
        pct_maxgap_ge3=("max_gap", lambda s: float((s >= 3).mean())),
        pct_maxgap_ge5=("max_gap", lambda s: float((s >= 5).mean())),
        pct_scattered=("max_gap", lambda s: float((s <= 2).mean())),  # incl. fully-covered
    ).round(2))
else:
    gaps = None
    print("skipped (no EE)")

In [ ]:
# --- Visualise gap structure + interpolation recommendation ---
if EE_OK and gaps is not None:
    # bucket each parcel-year by its longest missing run
    def bucket(g):
        if g == 0:
            return "0 (full)"
        if g <= 2:
            return "1-2 (scattered)"
        if g <= 4:
            return "3-4 (season chunk)"
        return "5+ (half-year+)"

    order = ["0 (full)", "1-2 (scattered)", "3-4 (season chunk)", "5+ (half-year+)"]
    colors = ["#1b9e77", "#7fbf7b", "#fdae61", "#d7191c"]
    gaps["bucket"] = gaps["max_gap"].map(bucket)
    tab = (
        gaps.groupby(["year", "bucket"]).size().unstack(fill_value=0)
        .reindex(columns=order, fill_value=0)
    )
    tab_frac = tab.div(tab.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(8, 4.2))
    bottom = np.zeros(len(tab_frac))
    for b, c in zip(order, colors):
        ax.bar(tab_frac.index.astype(str), tab_frac[b], bottom=bottom, label=b, color=c)
        bottom += tab_frac[b].values
    ax.set(xlabel="crop year", ylabel="fraction of parcels",
           title="Longest consecutive missing-month run (max_gap) per parcel-year")
    ax.legend(title="max_gap", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(OUT / "landsat_gap_structure_piura.png", bbox_inches="tight")
    plt.show()

    # parcel-weighted summary across all sampled years
    w = label_years.reindex(sorted(gaps["year"].unique())).fillna(0)
    per_year = gaps.groupby("year")["max_gap"]
    wt = w / w.sum()
    p_scattered = float(sum(wt[y] * (per_year.get_group(y) <= 2).mean() for y in wt.index))
    p_chunk = float(sum(wt[y] * (per_year.get_group(y) >= 3).mean() for y in wt.index))
    p_half = float(sum(wt[y] * (per_year.get_group(y) >= 5).mean() for y in wt.index))
    print("=== GAP-STRUCTURE VERDICT (parcel-weighted) ===")
    print(f"max_gap <= 2 (interpolate safely)      : {p_scattered:.0%} of parcels")
    print(f"max_gap >= 3 (contiguous season gap)   : {p_chunk:.0%} of parcels")
    print(f"max_gap >= 5 (half-year+ unobserved)   : {p_half:.0%} of parcels")
    print()
    if p_chunk < 0.25:
        print("-> Monthly grid + light linear/harmonic interpolation is safe for most parcels.")
    elif p_half < 0.20:
        print("-> Gaps are often contiguous season chunks, not scatter. Prefer BI-MONTHLY/SEASONAL")
        print("   composites (each cell stays a real measurement); reserve interpolation for the")
        print("   <=2-month scattered gaps; carry a `was_interpolated` mask channel.")
    else:
        print("-> Large contiguous gaps are common: do NOT interpolate across them. Bin SEASONALLY,")
        print("   use mask-native models (PSE-LTAE/Transformer), and ABSTAIN on max_gap>=5 parcels.")
    print("Note: best-pixel presence, so these are optimistic; true per-parcel gaps are larger.")
else:
    print("skipped (no EE)")

### 5c. Abstain-gate sensitivity — parcels retained vs (min `n_valid_obs`, max `max_gap`)

The `plan.md` §7 abstain gate keeps a parcel only if `n_valid_obs ≥ 6` **and** `max_gap < 5`
(≤ 4). That gate trades **label quality for training-set size**, so we sweep both knobs and estimate
the **total real-crop single-label parcels retained** for each combination.

Method: sample parcels **in every probed year** (best-pixel `n_valid_obs` = clear acquisitions/yr,
and `max_gap` from monthly presence, as §5/§5b), compute the retained *fraction* per year for each
threshold pair, then scale by that year's full parcel count and sum. Caveats: best-pixel metrics →
**optimistic upper bounds**; this sweeps only the two coverage knobs (the separate `0.09–50 ha` area
gate removes a further ~16 % sub-pixel parcels); base excludes years with < 30 labels (~4 %).

In [ ]:
# --- Sample per-parcel (n_valid_obs, max_gap) across ALL probed years for the gate sweep ---
GATE_SAMPLE = 150
if EE_OK:
    rng3 = np.random.default_rng(11)
    grows = []
    for y in PROBE_YEARS:
        pool = crop_single[crop_single["year"] == y]
        take = min(GATE_SAMPLE, len(pool))
        sample = pool.iloc[rng3.choice(len(pool), take, replace=False)]
        fc = to_fc(sample)
        coll = masked_coll(y).select("val").map(lambda im: im.float())

        obs_img = coll.count().unmask(0).rename("valid_obs")  # clear acquisitions / yr (best pixel)
        month_bands = []
        for m in range(1, 13):
            m0 = ee.Date.fromYMD(y, m, 1)
            month_bands.append(
                coll.filterDate(m0, m0.advance(1, "month"))
                .merge(BASE).sum().unmask(0).gt(0).rename(f"m{m:02d}")
            )
        stack = obs_img.addBands(ee.Image.cat(month_bands))
        red = stack.reduceRegions(collection=fc, reducer=ee.Reducer.max(), scale=30).getInfo()
        for f in red["features"]:
            p = f["properties"]
            pres = np.array([p.get(f"m{m:02d}", 0) or 0 for m in range(1, 13)], float)
            _, max_gap, _ = gap_metrics(pres)
            grows.append({"year": y, "valid_obs": int(p.get("valid_obs", 0) or 0), "max_gap": max_gap})
        print(f"  sampled {y}: n={take}")
    gate_sample = pd.DataFrame(grows)
    display(gate_sample.groupby("year").agg(
        n=("valid_obs", "size"),
        median_valid_obs=("valid_obs", "median"),
        median_max_gap=("max_gap", "median"),
    ))
else:
    gate_sample = None
    print("skipped (no EE)")

In [ ]:
# --- Sweep the gate: estimated total parcels retained for each (min n_valid_obs, max_gap) ---
MIN_OBS_GRID = [3, 4, 5, 6, 8, 10]      # keep if n_valid_obs >= this
MAXGAP_KEEP = [2, 3, 4, 5, 12]          # keep if max_gap <= this (12 = no gap constraint)

if EE_OK and gate_sample is not None:
    full_count = label_years.reindex(PROBE_YEARS).fillna(0)
    total_base = int(full_count.sum())

    grid = np.zeros((len(MIN_OBS_GRID), len(MAXGAP_KEEP)))
    for i, mo in enumerate(MIN_OBS_GRID):
        for j, mg in enumerate(MAXGAP_KEEP):
            est = 0.0
            for y in PROBE_YEARS:
                s = gate_sample[gate_sample["year"] == y]
                if len(s) == 0:
                    continue
                frac = float(((s["valid_obs"] >= mo) & (s["max_gap"] <= mg)).mean())
                est += frac * float(full_count[y])
            grid[i, j] = est

    grid_df = pd.DataFrame(
        grid.round().astype(int),
        index=[f"obs>={m}" for m in MIN_OBS_GRID],
        columns=[(f"gap<={g}" if g < 12 else "gap:any") for g in MAXGAP_KEEP],
    )
    print(f"base = {total_base:,} real-crop single-label parcels in probed years "
          f"(of {len(crop_single):,} total; no gate = 100%)")
    print("\nEstimated parcels retained:")
    display(grid_df)

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    im = ax.imshow(grid, aspect="auto", cmap="YlGn", vmin=0, vmax=total_base)
    ax.set_xticks(range(len(MAXGAP_KEEP)))
    ax.set_xticklabels(grid_df.columns)
    ax.set_yticks(range(len(MIN_OBS_GRID)))
    ax.set_yticklabels(grid_df.index)
    ax.set(xlabel="max_gap allowed (months)", ylabel="min n_valid_obs",
           title="Estimated parcels surviving abstain gate\n(n_valid_obs >= row  AND  max_gap <= col)")
    for i in range(len(MIN_OBS_GRID)):
        for j in range(len(MAXGAP_KEEP)):
            ax.text(j, i, f"{int(grid[i, j]):,}\n{grid[i, j] / total_base:.0%}",
                    ha="center", va="center", fontsize=7,
                    color="black" if grid[i, j] < 0.6 * total_base else "white")
    fig.colorbar(im, label="parcels retained")
    fig.tight_layout()
    fig.savefig(OUT / "abstain_gate_grid.png", bbox_inches="tight")
    plt.show()

    # highlight the plan.md §7 default gate (obs>=6 AND max_gap<=4)
    est_default = sum(
        float(((gate_sample[gate_sample["year"] == y]["valid_obs"] >= 6)
                & (gate_sample[gate_sample["year"] == y]["max_gap"] <= 4)).mean())
        * float(full_count[y])
        for y in PROBE_YEARS if (gate_sample["year"] == y).any()
    )
    print(f"\nplan.md §7 default gate (n_valid_obs>=6 AND max_gap<=4): "
          f"~{int(round(est_default)):,} parcels ({est_default / total_base:.0%} of probed base)")
    print("After the separate area gate (0.09-50 ha, ~16% sub-pixel): "
          f"~{int(round(est_default * 0.84)):,} parcels.")
    print("Optimistic (best-pixel) upper bound — true retention is somewhat lower.")
else:
    print("skipped (no EE)")